In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [39]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)
answer = assistant.rag('How do I run Ollama locally?')
# print(answer)
print('test')

test


In [40]:
question = 'I just discovered the course. Can I join it?'
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

# response = openai_client.responses.create(
#     model='gpt-5.4-mini',
#     input=messages,
# )

# response.output_text

In [41]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [42]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [43]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [47]:
call = response.output[0]
print(call)
import json
args = json.loads(call.arguments)
args

ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered the course enroll after start"}', call_id='call_3pc4Ka2MPldyqupaHfzjapad', name='search', type='function_call', id='fc_0cbd2f90b7a9c338006a380d2dcf0c81919dab4b5eb7596e33', namespace=None, status='completed')


{'query': 'Can I join the course late discovered the course enroll after start'}

In [48]:
call.name

'search'

In [18]:
results = search(**args)
result_json = json.dumps(results, indent=2)

In [34]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [35]:
messages.append(call)
messages.append(function_call_output)

In [36]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)
response.output_text

'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.'

make a call to LLM...
LLM decided to invode search(params)
we invoke the search, we have the results
send the results back to LLM (another call)
LLM processes the results
LLM gives the answer

In [51]:
def make_call(call):
    import json
    args = json.loads(call.arguments)
    
    if call.name == 'search':
        results = search(**args)
        result_json = json.dumps(results, indent=2)

        return {
            "type": "function_call_output",
            'call_id': call.call_id,
            'output': result_json,
        }
    # else:
    #     raise ValueError(f"Unknown tool: {call.name}")

In [59]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. Firstperfoem search, analyze the results, and then make another search if necessary.

The question has to be about the course or it's,logistics.
if the search returns nothing, it's likely an off-topic question. In that case, politely decline to answer and suggest the user to ask course related questions.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [57]:
def agent_loop(instructions, question, model='gpt-5.4-mini'):
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    i = 1

    while True:
        print(f'=== Iteration {i} ===')
        has_function_call = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )
        messages.extend(response.output)
        print(messages)

        for item in response.output:
            print('Received item:', item)
            if item.type == 'function_call':
                print('Function call detected:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_call = True
                
            elif item.type == 'message':
                answer = item.content[0].text
                print('Model response:', item.content[0].text)
        if not has_function_call:
            break
        i += 1
    return answer

In [60]:
question = "what's queen gambit?"
agent_loop(instructions, question)

=== Iteration 1 ===
[{'role': 'developer', 'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. Firstperfoem search, analyze the results, and then make another search if necessary.\n\nThe question has to be about the course or it's,logistics.\nif the search returns nothing, it's likely an off-topic question. In that case, politely decline to answer and suggest the user to ask course related questions.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"}, {'role': 'user', 'content': "what's queen gambit?"}, ResponseFunctionToolCall(arguments='{"query":"queen gambit chess queen\'s gambit"}', call_id='call_wBgRCuWn6rRf

'That looks like an off-topic question — I couldn’t find anything course-related about “queen gambit” in the course FAQ.\n\nIf you have a course-related question, feel free to ask and I’ll help. Is there anything else in the course you want to explore?'

In [61]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [62]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [63]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [64]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [65]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [66]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)
result

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. Firstperfoem search, analyze the results, and then make another search if necessary.\n\nThe question has to be about the course or it's,logistics.\nif the search returns nothing, it's likely an off-topic question. In that case, politely decline to answer and suggest the user to ask course related questions.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments

In [68]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. Firstperfoem search, analyze the results, and then make another search if necessary.\n\nThe question has to be about the course or it's,logistics.\nif the search returns nothing, it's likely an off-topic question. In that case, politely decline to answer and suggest the user to ask course related questions.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run 

In [69]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)
result2

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content='How do I run a different model?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Ollama run different model llama3 mistral course FAQ model name ollama run"}', call_id='call_4IXBZ325ctOzpXsyTfTCRM9Q', name='search', type='function_call', id='fc_0c15aa47fd80db86006a381ef4376881a0a7fe76c49389b9a8', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_4IXBZ325ctOzpXsyTfTCRM9Q', 'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL htt